# CropSure Research Notebook 06: Bayesian Trust Intelligence Engine & Anomaly Detection

## Context: Data Trust & Fraud Prevention in Agritech
Crowdsourced agricultural monitoring and satellite/mobile crop insurance verification are susceptible to data tampering, spoofing, and fraud. Malicious actors or automated bots may submit falsified foliage scans to illegitimately trigger insurance payouts, or faulty GPS hardware may submit erratic readings that skew regional epidemiological models.

Rather than treating disease predictions in isolation, **CropSure** incorporates a **Bayesian Trust Intelligence Engine** that synthesizes **12 multi-sensor evidence signals** into a calibrated probabilistic trust score ($0.0 - 100.0$) using conjugate Beta priors with exponential time decay, paired with strict kinematic and biological anomaly detectors.

## Research Objectives
1. Implement a Beta-conjugate Bayesian belief updating engine across 12 distinct verification dimensions.
2. Introduce exponential time decay ($\\lambda = 0.95$) to model the natural aging and staleness of historical evidence.
3. Build dedicated rule-based and kinematic anomaly detectors:
   - **Geospatial Teleportation Detector**: Flags travel velocities exceeding $100\\text{ km/h}$.
   - **Submission Frequency Detector**: Flags spam rates exceeding $20\\text{ scans/hour}$.
   - **Biological Severity Drop Detector**: Flags impossible recoveries exceeding $30\\%$ drop in $24\\text{ hours}$ without verified remediation.
4. Execute an automated test suite verifying mathematical bounds, convergence, and edge cases.
5. Package the engine for runtime integration with the CropSure Flask REST API (`/api/trust`).

In [ ]:
# Cell 02: Core Environment Imports
import os
import sys
import time
import math
import json
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

%matplotlib inline
print(f"[IMPORT] NumPy: {np.__version__} | SciPy: {sys.modules['scipy'].__version__}")

## Theoretical Foundation: Beta-Conjugate Updating Model

Each evidence signal $i \in \{1, \dots, 12\}$ models the latent probability $\theta_i \in [0, 1]$ that the incoming signal is authentic and uncompromised.

### 1. Conjugate Prior and Likelihood
We model $\theta_i$ with a Beta prior distribution:
$$P(\theta_i) = \text{Beta}(\alpha_i, \beta_i) = \frac{1}{\text{B}(\alpha_i, \beta_i)} \theta_i^{\alpha_i - 1} (1 - \theta_i)^{\beta_i - 1}$$

Given a continuous observation score $x \in [0.0, 1.0]$ with evidence weight $w > 0$, the Bayesian posterior update is:
$$\alpha_i^{\text{new}} = \alpha_i + w \cdot x$$
$$\beta_i^{\text{new}} = \beta_i + w \cdot (1 - x)$$

### 2. Posterior Mean & Epistemic Uncertainty
The expected trust score and its variance are analytically defined as:
$$\mathbb{E}[\theta_i] = \frac{\alpha_i}{\alpha_i + \beta_i}, \quad \text{Var}(\theta_i) = \frac{\alpha_i \beta_i}{(\alpha_i + \beta_i)^2 (\alpha_i + \beta_i + 1)}$$

### 3. Exponential Time Decay Dynamics
To prevent historical evidence from indefinitely dominating recent observations, parameters decay toward the uninformative baseline $(\alpha=1, \beta=1)$ over elapsed time $\Delta t$ with half-life decay parameter $\lambda = 0.95$:
$$\alpha_i(t + \Delta t) = 1.0 + \lambda^{\Delta t} (\alpha_i(t) - 1.0)$$
$$\beta_i(t + \Delta t) = 1.0 + \lambda^{\Delta t} (\beta_i(t) - 1.0)$$

In [ ]:
# Cell 04: BayesianTrustEngine Core Implementation
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path(os.path.abspath("")).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

class BayesianTrustEngine:
    """
    Bayesian Trust Verification Engine with 12 Evidence Signals and Beta Conjugacy.
    """
    SIGNALS = [
        "image_quality",
        "gps_consistency",
        "temporal_consistency",
        "disease_severity_plausibility",
        "weather_correlation",
        "historical_pattern",
        "device_fingerprint",
        "submission_frequency",
        "metadata_completeness",
        "expert_agreement",
        "cross_validation",
        "community_consensus"
    ]
    
    DEFAULT_WEIGHTS = {
        "image_quality": 0.12,
        "gps_consistency": 0.12,
        "temporal_consistency": 0.08,
        "disease_severity_plausibility": 0.12,
        "weather_correlation": 0.10,
        "historical_pattern": 0.08,
        "device_fingerprint": 0.06,
        "submission_frequency": 0.08,
        "metadata_completeness": 0.06,
        "expert_agreement": 0.08,
        "cross_validation": 0.05,
        "community_consensus": 0.05
    }
    
    def __init__(self, decay_rate: float = 0.95):
        self.decay_rate = decay_rate
        self.weights = self.DEFAULT_WEIGHTS.copy()
        # Normalize weights to sum to 1.0
        total_w = sum(self.weights.values())
        self.weights = {k: v / total_w for k, v in self.weights.items()}
        
        # Initialize Beta(alpha=2.0, beta=2.0) uninformative balanced priors
        self.state = {
            sig: {"alpha": 2.0, "beta": 2.0, "last_updated": datetime.utcnow().isoformat()}
            for sig in self.SIGNALS
        }
        
    def update_signal(self, signal_name: str, observation: float, weight: float = 1.0, timestamp: datetime = None):
        """
        Bayesian updating for an individual signal observation in [0, 1].
        """
        if signal_name not in self.SIGNALS:
            raise ValueError(f"Unknown evidence signal: {signal_name}")
            
        obs_clamped = max(0.0, min(1.0, float(observation)))
        curr = self.state[signal_name]
        
        # Beta update
        curr["alpha"] += weight * obs_clamped
        curr["beta"] += weight * (1.0 - obs_clamped)
        curr["last_updated"] = (timestamp or datetime.utcnow()).isoformat()
        
    def apply_time_decay(self, elapsed_days: float = 1.0):
        """
        Applies exponential decay toward non-informative prior Beta(1, 1).
        """
        decay_factor = self.decay_rate ** elapsed_days
        for sig in self.SIGNALS:
            curr = self.state[sig]
            curr["alpha"] = 1.0 + decay_factor * (curr["alpha"] - 1.0)
            curr["beta"] = 1.0 + decay_factor * (curr["beta"] - 1.0)
            
    def calculate_trust_score(self) -> dict:
        """
        Aggregates all 12 signals into composite trust score [0, 100].
        """
        signal_scores = {}
        signal_variances = {}
        weighted_score_sum = 0.0
        weighted_var_sum = 0.0
        
        for sig in self.SIGNALS:
            a = self.state[sig]["alpha"]
            b = self.state[sig]["beta"]
            expected_val = a / (a + b)
            variance = (a * b) / (((a + b) ** 2) * (a + b + 1.0))
            
            signal_scores[sig] = round(expected_val * 100.0, 2)
            signal_variances[sig] = round(variance, 6)
            
            w = self.weights[sig]
            weighted_score_sum += w * expected_val
            weighted_var_sum += (w ** 2) * variance
            
        composite_score = round(weighted_score_sum * 100.0, 2)
        composite_uncertainty = round(math.sqrt(weighted_var_sum) * 100.0, 2)
        
        # Categorical classification
        if composite_score >= 70.0:
            status = "TRUSTED"
        elif composite_score >= 40.0:
            status = "SUSPICIOUS"
        else:
            status = "REJECTED"
            
        return {
            "trust_score": composite_score,
            "uncertainty": composite_uncertainty,
            "classification": status,
            "signal_breakdown": signal_scores,
            "signal_variances": signal_variances
        }

print("[ENGINE] BayesianTrustEngine class instantiated successfully.")

In [ ]:
# Cell 05: Anomaly Detectors (Kinematic, Rate, and Biological Kinetic)
def haversine_distance_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculates great-circle distance between two geographic coordinates in kilometers."""
    R = 6371.0 # Earth radius in km
    d_lat = math.radians(lat2 - lat1)
    d_lon = math.radians(lon2 - lon1)
    a = (
        math.sin(d_lat / 2.0) ** 2 +
        math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(d_lon / 2.0) ** 2
    )
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c

class GeoSpatialJumpDetector:
    """Flags impossible physical teleportation (> 100 km in < 1 hour)."""
    MAX_SPEED_KMH = 100.0
    
    @classmethod
    def check(cls, prev_scan: dict, curr_scan: dict) -> dict:
        if not prev_scan or "latitude" not in prev_scan or "latitude" not in curr_scan:
            return {"is_anomaly": False, "speed_kmh": 0.0}
            
        dist_km = haversine_distance_km(
            prev_scan["latitude"], prev_scan["longitude"],
            curr_scan["latitude"], curr_scan["longitude"]
        )
        t_prev = datetime.fromisoformat(prev_scan["timestamp"])
        t_curr = datetime.fromisoformat(curr_scan["timestamp"])
        elapsed_hours = max((t_curr - t_prev).total_seconds() / 3600.0, 1e-4)
        speed = dist_km / elapsed_hours
        
        is_jump = speed > cls.MAX_SPEED_KMH and dist_km > 5.0
        return {
            "is_anomaly": bool(is_jump),
            "detector": "GeoSpatialJumpDetector",
            "distance_km": round(dist_km, 2),
            "elapsed_hours": round(elapsed_hours, 3),
            "calculated_speed_kmh": round(speed, 2)
        }

class FrequencyDetector:
    """Flags automated bot submission bursts (> 20 scans per hour)."""
    MAX_SCANS_PER_HOUR = 20
    
    @classmethod
    def check(cls, recent_timestamps: list, current_timestamp: datetime) -> dict:
        one_hour_ago = current_timestamp - timedelta(hours=1)
        scans_last_hour = sum(1 for ts in recent_timestamps if ts >= one_hour_ago)
        is_burst = scans_last_hour > cls.MAX_SCANS_PER_HOUR
        return {
            "is_anomaly": bool(is_burst),
            "detector": "FrequencyDetector",
            "scans_in_window": scans_last_hour,
            "threshold": cls.MAX_SCANS_PER_HOUR
        }

class SeverityDropDetector:
    """Flags biologically impossible recovery (> 30% severity drop in 24h without verified fungicide)."""
    MAX_UNREMEDIATED_DROP = 0.30
    
    @classmethod
    def check(cls, prev_severity: float, curr_severity: float, elapsed_hours: float, treated: bool = False) -> dict:
        drop = prev_severity - curr_severity
        is_impossible = (drop > cls.MAX_UNREMEDIATED_DROP) and (elapsed_hours <= 24.0) and (not treated)
        return {
            "is_anomaly": bool(is_impossible),
            "detector": "SeverityDropDetector",
            "severity_drop": round(drop, 4),
            "elapsed_hours": round(elapsed_hours, 2),
            "treatment_applied": treated
        }

print("[ANOMALY] GeoSpatialJump, Frequency, and SeverityDrop detectors compiled.")

In [ ]:
# Cell 06: Trust Engine Execution Demonstration (Honest Farmer vs Malicious Actor)
engine_honest = BayesianTrustEngine()
engine_fraud = BayesianTrustEngine()

# Scenario 1: Authentic, diligent smallholder farmer
honest_observations = {
    "image_quality": 0.92,
    "gps_consistency": 0.98,
    "temporal_consistency": 0.95,
    "disease_severity_plausibility": 0.88,
    "weather_correlation": 0.85,
    "historical_pattern": 0.90,
    "device_fingerprint": 0.96,
    "submission_frequency": 0.90,
    "metadata_completeness": 0.94,
    "expert_agreement": 0.89,
    "cross_validation": 0.85,
    "community_consensus": 0.88
}

# Scenario 2: Spoofed / bot submissions
fraud_observations = {
    "image_quality": 0.25,
    "gps_consistency": 0.10,
    "temporal_consistency": 0.30,
    "disease_severity_plausibility": 0.15,
    "weather_correlation": 0.20,
    "historical_pattern": 0.10,
    "device_fingerprint": 0.20,
    "submission_frequency": 0.05,
    "metadata_completeness": 0.30,
    "expert_agreement": 0.10,
    "cross_validation": 0.20,
    "community_consensus": 0.15
}

# Apply updates over 5 sequential scans
for _ in range(5):
    for sig, val in honest_observations.items():
        engine_honest.update_signal(sig, val, weight=1.0)
    for sig, val in fraud_observations.items():
        engine_fraud.update_signal(sig, val, weight=1.0)

res_honest = engine_honest.calculate_trust_score()
res_fraud = engine_fraud.calculate_trust_score()

print("==================================================")
print("          Bayesian Trust Scoring Benchmark        ")
print("==================================================")
print(f" Honest Farmer Profile  : Score = {res_honest['trust_score']:.1f}/100 | Status = {res_honest['classification']}")
print(f" Fraudulent Bot Profile : Score = {res_fraud['trust_score']:.1f}/100 | Status = {res_fraud['classification']}")
print("==================================================")

In [ ]:
# Cell 07: Visualizations (Trust Evolution & 12-Signal Radar Chart)
# 1. Trust Score Trajectory Over 20 Sequential Scans
sim_steps = 20
eng_h = BayesianTrustEngine()
eng_f = BayesianTrustEngine()

honest_scores = []
fraud_scores = []

for step in range(sim_steps):
    for s, v in honest_observations.items():
        eng_h.update_signal(s, v + np.random.normal(0, 0.04))
    for s, v in fraud_observations.items():
        eng_f.update_signal(s, v + np.random.normal(0, 0.04))
        
    honest_scores.append(eng_h.calculate_trust_score()["trust_score"])
    fraud_scores.append(eng_f.calculate_trust_score()["trust_score"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Trajectory Plot
axes[0].plot(range(1, sim_steps + 1), honest_scores, 'o-', color='#2ca02c', lw=2.5, label='Honest Farmer Profile')
axes[0].plot(range(1, sim_steps + 1), fraud_scores, 's--', color='#d62728', lw=2.5, label='Fraudulent / Bot Profile')
axes[0].axhline(70.0, color='green', linestyle=':', label='Trusted Threshold (>=70)')
axes[0].axhline(40.0, color='red', linestyle=':', label='Rejected Threshold (<40)')
axes[0].set_title('Bayesian Trust Evolution across Sequential Submissions', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Scan Submission Sequence')
axes[0].set_ylabel('Composite Trust Score [0 - 100]')
axes[0].set_ylim(0, 105)
axes[0].legend(loc='center right')
axes[0].grid(True, alpha=0.3)

# 2. Radar Chart across 12 Evidence Dimensions
categories = [s.replace('_', '\n') for s in BayesianTrustEngine.SIGNALS]
N_cats = len(categories)
angles = [n / float(N_cats) * 2 * math.pi for n in range(N_cats)]
angles += angles[:1]

val_h = [res_honest["signal_breakdown"][s] for s in BayesianTrustEngine.SIGNALS]
val_h += val_h[:1]
val_f = [res_fraud["signal_breakdown"][s] for s in BayesianTrustEngine.SIGNALS]
val_f += val_f[:1]

axes[1].remove() # Replace with polar projection
ax_radar = fig.add_subplot(1, 2, 2, polar=True)

ax_radar.plot(angles, val_h, color='#2ca02c', linewidth=2, linestyle='solid', label='Honest Farmer')
ax_radar.fill(angles, val_h, color='#2ca02c', alpha=0.25)

ax_radar.plot(angles, val_f, color='#d62728', linewidth=2, linestyle='solid', label='Fraudulent Actor')
ax_radar.fill(angles, val_f, color='#d62728', alpha=0.25)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(categories, fontsize=8)
ax_radar.set_ylim(0, 100)
ax_radar.set_title('12-Signal Verification Radar Profile', fontsize=12, fontweight='bold', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.25, 1.15))

plt.tight_layout()
plt.show()

In [ ]:
# Cell 08: Comprehensive Automated Unit Test Suite
print("[TESTS] Executing test suite for Bayesian Trust Engine & Anomaly Detectors...")

# Test 1: Prior initialization across all 12 signals
test_eng = BayesianTrustEngine()
assert len(test_eng.state) == 12, "Failed: Engine missing some of the 12 signals!"
for s, v in test_eng.state.items():
    assert v["alpha"] == 2.0 and v["beta"] == 2.0, f"Failed: Signal {s} initial prior incorrect!"

# Test 2: Trust score bounds [0.0, 100.0]
init_score = test_eng.calculate_trust_score()["trust_score"]
assert 0.0 <= init_score <= 100.0, "Failed: Score out of bounds!"
assert init_score == 50.0, f"Failed: Initial balanced prior should equal 50.0, got {init_score}"

# Test 3: Geospatial teleportation detection (>100 km in 30 mins)
scan1 = {"latitude": 28.6139, "longitude": 77.2090, "timestamp": "2026-10-14T10:00:00"} # New Delhi
scan2 = {"latitude": 19.0760, "longitude": 72.8777, "timestamp": "2026-10-14T10:30:00"} # Mumbai (~1150 km away in 30m)
jump_check = GeoSpatialJumpDetector.check(scan1, scan2)
assert jump_check["is_anomaly"] is True, "Failed: Should detect 1150km jump in 30 mins as anomaly!"

# Test 4: Rate limit frequency detection (>20 scans in 1 hour)
now = datetime.utcnow()
timestamps = [now - timedelta(minutes=i) for i in range(25)]
freq_check = FrequencyDetector.check(timestamps, now)
assert freq_check["is_anomaly"] is True, "Failed: 25 scans in 1h must trigger rate limit!"

# Test 5: Severity drop biological impossibility (>30% in 24h without treatment)
drop_check = SeverityDropDetector.check(prev_severity=0.80, curr_severity=0.40, elapsed_hours=12.0, treated=False)
assert drop_check["is_anomaly"] is True, "Failed: 40% drop without treatment should be flagged!"
drop_remediated = SeverityDropDetector.check(prev_severity=0.80, curr_severity=0.40, elapsed_hours=12.0, treated=True)
assert drop_remediated["is_anomaly"] is False, "Failed: Remediated drop should not be flagged!"

# Test 6: Time decay convergence toward uninformative prior
decay_eng = BayesianTrustEngine(decay_rate=0.90)
decay_eng.update_signal("image_quality", 1.0, weight=10.0)
before_decay = decay_eng.state["image_quality"]["alpha"]
decay_eng.apply_time_decay(elapsed_days=30.0)
after_decay = decay_eng.state["image_quality"]["alpha"]
assert after_decay < before_decay, "Failed: Alpha must decay toward 1.0 over time!"

print("\n==================================================")
print("   ALL 6 UNIT TESTS PASSED WITH 100% COMPLIANCE   ")
print("==================================================")

In [ ]:
# Cell 09: CropSure REST API Integration Helper & State Persistence
def evaluate_scan_trust(current_scan: dict, previous_scan: dict = None, engine: BayesianTrustEngine = None) -> dict:
    """
    Enterprise helper function for CropSure REST API (/api/trust).
    Executes anomaly checks, updates Bayesian signals, and returns JSON-serializable assessment.
    """
    if engine is None:
        engine = BayesianTrustEngine()
        
    anomalies_detected = []
    
    # 1. Kinematic Geolocation Check
    if previous_scan:
        geo_res = GeoSpatialJumpDetector.check(previous_scan, current_scan)
        if geo_res["is_anomaly"]:
            anomalies_detected.append(geo_res)
            engine.update_signal("gps_consistency", 0.0, weight=2.0)
        else:
            engine.update_signal("gps_consistency", 1.0, weight=1.0)
            
    # 2. Biological Kinetics Check
    if previous_scan and "severity" in previous_scan and "severity" in current_scan:
        t_prev = datetime.fromisoformat(previous_scan["timestamp"])
        t_curr = datetime.fromisoformat(current_scan["timestamp"])
        hours = max((t_curr - t_prev).total_seconds() / 3600.0, 0.1)
        drop_res = SeverityDropDetector.check(
            prev_severity=previous_scan["severity"],
            curr_severity=current_scan["severity"],
            elapsed_hours=hours,
            treated=current_scan.get("treated", False)
        )
        if drop_res["is_anomaly"]:
            anomalies_detected.append(drop_res)
            engine.update_signal("disease_severity_plausibility", 0.0, weight=2.0)
        else:
            engine.update_signal("disease_severity_plausibility", 0.95, weight=1.0)
            
    # 3. Direct evidence assimilation
    for sig in ["image_quality", "temporal_consistency", "metadata_completeness"]:
        if sig in current_scan:
            engine.update_signal(sig, current_scan[sig], weight=1.0)
            
    assessment = engine.calculate_trust_score()
    assessment["anomalies"] = anomalies_detected
    assessment["is_clean"] = len(anomalies_detected) == 0
    assessment["timestamp"] = datetime.utcnow().isoformat()
    return assessment

# Export initial state configuration to models/bayesian_trust_state.json
state_path = MODELS_DIR / "bayesian_trust_state.json"
with open(state_path, "w", encoding="utf-8") as f:
    json.dump({"state": engine_honest.state, "weights": engine_honest.weights}, f, indent=2)

print(f"[EXPORT] Trust engine baseline state saved to: {state_path}")

# Demonstration API Call
demo_scan = {
    "latitude": 28.6139,
    "longitude": 77.2090,
    "timestamp": datetime.utcnow().isoformat(),
    "image_quality": 0.94,
    "temporal_consistency": 0.90,
    "metadata_completeness": 0.98,
    "severity": 0.35,
    "treated": False
}
api_res = evaluate_scan_trust(demo_scan)
print(json.dumps(api_res, indent=2))

## Summary, Operational Constraints & Future Extensions

### Framework Summary
The Bayesian Trust Intelligence Engine guarantees that agricultural disease predictions are contextualized by data authenticity. By pairing 12-signal Beta-conjugate belief updating with physical anomaly detectors, the platform actively protects against automated spoofing, GPS corruption, and insurance claim fraud.

### Operational Constraints & Assumptions
1. **Cold-Start Bias**: For first-time app users, the uninformative prior $\\text{Beta}(2, 2)$ initializes trust at exactly $50.0$. Submissions from newly onboarded farmers should receive non-punitive provisional review until $N \\ge 3$ consistent scans are registered.
2. **GPS Foliar Attenuation**: Under dense multi-tier tomato greenhouse canopies, multipath satellite reflections can induce artificial coordinate drift ($10-30\\text{ meters}$). Speed calculation incorporates a $5\\text{ km}$ minimum spatial threshold before triggering teleportation warnings.

### Future Enhancements
- **Dirichlet-Multinomial Extension**: Generalize the scalar $[0, 1]$ Beta model into a multi-class Dirichlet distribution to assess categorical claims (e.g. soil type, fertilizer brand).
- **Federated Cooperative Verification**: Enable cryptographic peer validation across local farmer cooperative mesh networks.